# Clang AST generation

Clang was chosen over GCC for AST generation because it provides a stable and user-oriented AST API, making it more suitable for program analysis and machine learning workflows.

## Imports

In [ ]:
import sys
import json
from pathlib import Path
from gnn_ai_code_detector.preprocess import ClangASTConverter

## AST preparation

In [2]:
EXAMPLE = Path("./examples/add.c")
CLANG = r"C:\Program Files\LLVM\bin\clang.exe"

In [ ]:
converter = ClangASTConverter(CLANG)

To generate the AST in JSON format, Clang's -ast-dump=json functionality is invoked. The resulting tree contains a complete syntactic representation of the translation unit, including user-defined code and compiler-generated information.

In [4]:
ast = converter._build_ast(EXAMPLE)

output = Path("examples/add_ast.json")

with output.open("w", encoding="utf-8") as f:
    json.dump(ast, f, indent=2)

for branch in ast.get("inner"):
    print(branch["kind"], branch.get("name"))

RecordDecl _GUID
TypedefDecl __int128_t
TypedefDecl __uint128_t
TypedefDecl __NSConstantString
TypedefDecl size_t
TypedefDecl __builtin_ms_va_list
TypedefDecl __builtin_va_list
TypedefDecl uintptr_t
TypedefDecl va_list
FunctionDecl __va_start
FunctionDecl __va_start
TypedefDecl size_t
TypedefDecl ptrdiff_t
TypedefDecl intptr_t
TypedefDecl __vcrt_bool
TypedefDecl wchar_t
FunctionDecl __security_init_cookie
FunctionDecl __security_check_cookie
FunctionDecl __report_gsfailure
VarDecl __security_cookie
TypedefDecl __crt_bool
FunctionDecl _invalid_parameter_noinfo
FunctionDecl _invalid_parameter_noinfo_noreturn
FunctionDecl _invoke_watson
TypedefDecl errno_t
TypedefDecl wint_t
TypedefDecl wctype_t
TypedefDecl __time32_t
TypedefDecl __time64_t
RecordDecl __crt_locale_data_public
TypedefDecl __crt_locale_data_public
RecordDecl __crt_locale_pointers
TypedefDecl __crt_locale_pointers
TypedefDecl _locale_t
RecordDecl _Mbstatet
TypedefDecl _Mbstatet
TypedefDecl mbstate_t
TypedefDecl time_t
Typede

Besides the code present in the file itself, Clang's AST also contains the branches that represent the resolved includes, which get removed since they are irrelevant to the code structure and would present a very large amount of noise. 

In [5]:
cut_ast = converter._cut_irrelevant_branches(ast, Path("examples/add.c"))

output = Path("examples/add_cut_ast.json")

with output.open("w", encoding="utf-8") as f:
    json.dump(cut_ast, f, indent=2)

for branch in cut_ast["inner"]:
    print(branch["kind"], branch.get("name"))


FunctionDecl add
FunctionDecl main


Irrelevant metadata used for source mapping and debugging purposes such as node identifiers, source locations and source ranges gets removed for clarity.

In [6]:
clean_ast = converter._remove_metadata(cut_ast)

output = Path("examples/add_clean_ast.json")

with output.open("w", encoding="utf-8") as f:
    json.dump(clean_ast, f, indent=2)


Clang additionally inserts compiler-generated nodes that do not correspond to explicit constructs in the source code, which get flattened because they add redundancy without useful information.

In [7]:
clean_ast = converter._remove_compiler_artifacts(clean_ast)

output = Path("examples/add_ast_no_artifacts.json")

with output.open("w", encoding="utf-8") as f:
    json.dump(clean_ast, f, indent=2)


# HumanVsAI_CodeDataset

## Imports

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt

## Dataset exploration

In [8]:
df = pd.read_csv("../../data/Code_Dataset/HumanVsAi_CodeDataset.csv")
df.head()

,problem_id,Sample_Code,Generated,Language,Source
0,Prob3509,\n/* ITP1_2_D */\n\nimport Java.io.*;\nimport ...,Human,Java,CodeNet
1,Prob292,import Java.util.Scanner;\n\npublic class Main...,Human,Java,CodeNet
2,Prob4407,from datetime import datetime\n\ndef get_time_...,AI,Python,ChatGPT-4
3,Prob4805,"N,M = [int(i) for i in input().split()]\nc = [...",Human,Python,CodeNet
4,Prob5079,import Java.util.*;\n\n\npublic class Main {\n...,Human,Java,CodeNet


In [11]:
c_cpp_df = df[df["Language"].isin(["C", "C++"])].copy()

print(f"Total C/C++ samples: {len(c_cpp_df)}")

Total C/C++ samples: 4377


In [15]:
print(c_cpp_df["Generated"].value_counts())

Generated
AI       2189
Human    2188
Name: count, dtype: int64


In [20]:
print(f"Unique problems: {c_cpp_df["problem_id"].nunique()}")

Unique problems: 2847


In [ ]:
solution_counts = c_cpp_df.groupby("problem_id")["Generated"].value_counts().unstack(fill_value=0)

only_ai = ((solution_counts["AI"] > 0) & (solution_counts["Human"] == 0)).sum()
only_human = ((solution_counts["Human"] > 0) & (solution_counts["AI"] == 0)).sum()
both = ((solution_counts["AI"] > 0) & (solution_counts["Human"] > 0)).sum()

print(f"Only AI: {only_ai}")
print(f"Only Human: {only_human}")
print(f"Both: {both}")

Only AI: 1465
Only Human: 1382
Both: 0


The C/C++ subset is almost perfectly balanced with respect to authorship, containing 2189 AI-generated and 2188 human-written solutions. The dataset contains 2847 unique problems, with each problem represented by one or more solutions either completely AI-generated or human-written. Since multiple solutions may correspond to the same programming problem, the dataset should be partitioned using `problem_id` as the grouping variable to prevent data leakage.